In [1]:
import os
from dotenv import load_dotenv

# Load OPENAI_API_KEY from .env file in the project root
load_dotenv('../.env')
print('API key loaded:', bool(os.getenv('OPENAI_API_KEY')))

API key loaded: True


# LLM Agents for Financial Data — From Scratch

This notebook shows how to build a financial analysis agent step by step.

We start from the most basic LLM call and progressively add capabilities until we have
an agent that can answer real-time financial questions using live market data.

---

### Roadmap

| Step | What we build | Key concept |
|------|--------------|-------------|
| 1 | Plain LLM call | Understand the limitation: no real-time data |
| 2 | First tool (`get_stock_price`) | Give the LLM one ability |
| 3 | Richer toolset (8 tools) | More data sources → better answers |
| 4 | Full LangChain agent | LLM decides which tools to call |
| 5 | Agno agent (alternative framework) | Pre-built tools, less code |

> **Note:** `08_llm_interpretation.ipynb` applies this same pattern to our thesis forecasting models.

In [2]:
from langchain_openai import ChatOpenAI

# gpt-4o-mini: fast and cheap, good enough for tool-calling tasks
llm = ChatOpenAI(model="gpt-4o-mini")

/Users/amonterom/Documents/Financial-Forecast-Comparator/venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


## Step 1 — The Problem: A Plain LLM Has No Real-Time Data

Let's first see what happens when we ask a plain LLM about current stock prices.
A plain LLM only knows what was in its **training data** — it has a knowledge cutoff
and no access to live prices, news, or financial statements.

In [3]:
# Direct LLM call — no tools, no real-time data
# The LLM will admit it cannot answer because it has no access to live market data
llm.invoke("What is the stock price of Apple?")

AIMessage(content="I don't have real-time data access to provide the current stock price of Apple (AAPL). You can check financial news websites, stock market apps, or brokerage platforms for the latest stock price.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 15, 'total_tokens': 54, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_5e5eb987d2', 'id': 'chatcmpl-DSsv6EWedSbMUwsVZAdkqga31Pboy', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d7489-9756-76a0-acde-1a0a94b5cbe3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 15, 'output_tokens': 39, 'total_tokens': 54, 'input_token_details': {'audio': 0, 'cache_re

**Result:** The LLM refuses — it has no access to real-time data.

**Solution:** Give it **tools** — Python functions it can call to fetch live data on demand.
The LLM reads the tool's docstring to understand when to use it.

## Step 2 — First Tool: `get_stock_price`

A **tool** is just a Python function decorated with `@tool`.
The **docstring** is what the LLM reads to decide *when* to call it — write it clearly.

When the LLM is given tools, it becomes an **agent**: it can decide to call a tool,
observe the result, and use it to answer your question.

In [4]:
import yfinance as yf
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool

llm = ChatOpenAI(model="gpt-4o-mini")

# The @tool decorator registers this function as a tool the LLM can call.
# The docstring IS the tool description — the LLM reads it to decide when to use it.
@tool
def get_stock_price(symbol: str) -> str:
    """Get the current stock price of a company given its stock symbol (e.g. AAPL, TSLA)."""
    ticker = yf.Ticker(symbol)
    todays_data = ticker.history(period='1d')
    return f"The current stock price of {symbol} is ${todays_data['Close'].iloc[-1]:.2f}"

In [5]:
from langchain.tools import tool
from datetime import date

# --- Company overview ---
@tool
def company_information(ticker: str) -> dict:
    """Use this tool to retrieve company information like address, industry, sector, company officers,
    business summary, website, marketCap, current price, ebitda, total debt, total revenue, debt-to-equity, etc."""
    ticker_obj = yf.Ticker(ticker)
    return ticker_obj.get_info()

# --- Dividends & earnings calendar ---
@tool
def last_dividend_and_earnings_date(ticker: str) -> dict:
    """Use this tool to retrieve company's last dividend date and earnings release dates.
    It does not provide information about historical dividend yields."""
    ticker_obj = yf.Ticker(ticker)
    return ticker_obj.get_calendar()

# --- Ownership: mutual funds ---
@tool
def summary_of_mutual_fund_holders(ticker: str) -> dict:
    """Use this tool to retrieve company's top mutual fund holders.
    It also returns their percentage of share, stock count and value of holdings."""
    ticker_obj = yf.Ticker(ticker)
    mf_holders = ticker_obj.get_mutualfund_holders()
    return mf_holders.to_dict(orient="records")

# --- Ownership: institutional investors ---
@tool
def summary_of_institutional_holders(ticker: str) -> dict:
    """Use this tool to retrieve company's top institutional holders.
    It also returns their percentage of share, stock count and value of holdings."""
    ticker_obj = yf.Ticker(ticker)
    inst_holders = ticker_obj.get_institutional_holders()
    return inst_holders.to_dict(orient="records")

# --- Analyst ratings ---
@tool
def stock_grade_upgrades_downgrades(ticker: str) -> dict:
    """Use this to retrieve grade ratings upgrades and downgrades details of a particular stock.
    It'll provide name of firms along with 'To Grade' and 'From Grade' details. Grade date is also provided."""
    ticker_obj = yf.Ticker(ticker)
    curr_year = date.today().year
    upgrades_downgrades = ticker_obj.get_upgrades_downgrades()
    # Filter to current year only to keep the response concise
    upgrades_downgrades = upgrades_downgrades.loc[upgrades_downgrades.index > f"{curr_year}-01-01"]
    upgrades_downgrades = upgrades_downgrades[upgrades_downgrades["Action"].isin(["up", "down"])]
    return upgrades_downgrades.to_dict(orient="records")

# --- Corporate actions ---
@tool
def stock_splits_history(ticker: str) -> dict:
    """Use this tool to retrieve company's historical stock splits data."""
    ticker_obj = yf.Ticker(ticker)
    hist_splits = ticker_obj.get_splits()
    return hist_splits.to_dict()

# --- Latest news ---
@tool
def stock_news(ticker: str) -> dict:
    """Use this to retrieve latest news articles discussing a particular stock ticker."""
    ticker_obj = yf.Ticker(ticker)
    return ticker_obj.get_news()

print("Tools defined:", [
    company_information.name,
    last_dividend_and_earnings_date.name,
    summary_of_mutual_fund_holders.name,
    summary_of_institutional_holders.name,
    stock_grade_upgrades_downgrades.name,
    stock_splits_history.name,
    stock_news.name,
])

Tools defined: ['company_information', 'last_dividend_and_earnings_date', 'summary_of_mutual_fund_holders', 'summary_of_institutional_holders', 'stock_grade_upgrades_downgrades', 'stock_splits_history', 'stock_news']


## Step 3 — Richer Toolset: 7 More Financial Tools

One tool is a good start, but a useful financial agent needs more data sources.
We add 7 more tools covering company fundamentals, holders, analyst ratings, and news.

Each tool follows the same pattern:
1. Accept a `ticker` string as input
2. Call `yfinance` to fetch the data
3. Return it in a format the LLM can read and reason about

In [6]:
# Collect all tools into a single list
tools = [
    get_stock_price,
    company_information,
    last_dividend_and_earnings_date,
    summary_of_mutual_fund_holders,
    summary_of_institutional_holders,
    stock_grade_upgrades_downgrades,
    stock_splits_history,
    stock_news,
]

# Create the agent — the system_prompt sets its persona and behavior
finance_agent = create_agent(
    llm,
    tools=tools,
    system_prompt="You are a helpful financial assistant. Answer user queries using the available tools.",
)

print(f"Agent ready with {len(tools)} tools.")

Agent ready with 8 tools.


## Step 4 — Build the Agent

Now we wire the LLM and all tools together with `create_agent`.

The agent runs a **think → act → observe loop**:
1. Reads your question
2. Picks a tool to call
3. Gets the tool result
4. Decides if it needs more tools, or if it can answer
5. Returns the final response

You only write the question — the agent decides the rest.

In [7]:
query = "What is the last dividend date of Apple?"

# Invoke the agent — input must follow the messages format
response = finance_agent.invoke({"messages": [{"role": "user", "content": query}]})

# The final answer is always the last message in the response
print(response["messages"][-1].content)

The last dividend date for Apple (AAPL) is February 11, 2026, with an ex-dividend date of February 8, 2026.


## Step 5 — Query the Agent

Change the `query` below and rerun.
The agent will automatically pick the right tool(s) — you don't specify which ones.

## Alternative: Agno Framework

So far we built the agent manually with LangChain — defining every tool ourselves.

**Agno** (formerly `phi`) offers pre-built toolkits like `YFinanceTools` that wrap
the same `yfinance` calls for you. Less code, but less control over what the LLM sees.

| | LangChain (above) | Agno (below) |
|---|---|---|
| Tool definition | You write each `@tool` | Pre-built toolkit |
| Output control | Full control | Fixed format |
| Best for | Thesis / production | Quick prototyping |

> See `08_llm_interpretation.ipynb` for why custom tools are better for the thesis.

In [8]:
from agno.agent import Agent
from agno.models.openai import OpenAIChat
from agno.tools.yfinance import YFinanceTools

# Agno's Agent takes pre-built toolkits instead of individual @tool functions.
# YFinanceTools wraps yfinance calls — you enable the capabilities you need.
finance_agent = Agent(
    name="Finance Agent",
    role="Get financial data",
    model=OpenAIChat(id="gpt-4o-mini"),
    tools=[YFinanceTools(
        enable_stock_price=True,           # live price
        enable_analyst_recommendations=True,  # analyst ratings
        enable_company_info=True,          # fundamentals
        enable_company_news=True,          # latest headlines
    )],
    instructions=["Always use tables to display data"],
    markdown=True,  # format response as markdown
)

print("Agno finance agent ready!")

Agno finance agent ready!


In [9]:
# stream=True prints the response token by token as it arrives
finance_agent.print_response("Research NVDA and share analyst recommendations", stream=True)

Output()

### Agno queries

With Agno, use `print_response()` instead of `invoke()`.
It streams the output directly — no need to extract from a messages list.

In [10]:
# Historical price query — the agent will use get_historical_prices under the hood
finance_agent.print_response("How much did Nvidia stock increase in last 1 year?", stream=True)

Output()

In [11]:
# Fundamentals query — the agent will fetch company_info to find these metrics
query = "What are ebitda, total debt, total revenue and debt-to-equity for Nvidia stock?"
finance_agent.print_response(query, stream=True)

Output()